# Workers AI extraction with Clairic

`CloudflareWorkersAIExtractionModel` extracts typed values from text and images through Clairic's native API. This notebook shows an invoice, image-only extraction, and a combined Clairic + Clef example using nested Pydantic models.

Run `make notebook_check` and `make notebook_lab` from the repository root. Set `CF_ACCOUNT_ID` and `CF_AI_API_TOKEN` in the repo-root `.env`. Run the cells in order; they make live Workers AI calls. Outputs are cleared in the committed notebook.

[Clairic documentation](https://developers.cloudflare.com/workers-ai/models/clairic/) · [Decision-model notebook](decision_models.ipynb)


In [ ]:
# MARK: - SETUP
import base64
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_cloudflare import (
    CloudflareWorkersAIDecisionModel,
    CloudflareWorkersAIExtractionModel,
)

# Locate the checkout from either the repository root or docs/.
repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "libs/langchain-cloudflare/pyproject.toml").is_file()
)
load_dotenv(repo_root / ".env")
missing = [name for name in ("CF_ACCOUNT_ID", "CF_AI_API_TOKEN") if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Set these variables before running: {', '.join(missing)}")

# Run from the library directory to import its shared Worker example below.
os.chdir(repo_root / "libs/langchain-cloudflare")

# Direct REST keeps this example independent of any configured gateway.
model = CloudflareWorkersAIExtractionModel(ai_gateway=None)

## Text extraction

Fields use Clairic's native definitions. Here `text` copies source text, `number` extracts an amount, `date` calculates the due date, and `list` collects contact names. The missing purchase order can return `None`.

`extract()` returns the complete native response; no output-schema conversion or parsing is required.


In [ ]:
# MARK: - INVOICE
invoice = (
    "Invoice INV-2291 from Northwind Supply, issued 2026-09-02, net 30. "
    "4 pallets at $250.00 each, shipping $80.00. Contact: Dana Ruiz, Priya Shah."
)
fields = {
    "invoice_number": {"type": "text"},
    "due_date": {
        "type": "date",
        "instructions": "Issue date plus the net payment days",
    },
    "total": {
        "type": "number",
        "decimals": 2,
        "unit": "USD",
        "instructions": "Amount due including shipping",
    },
    "contacts": {"type": "list", "items": "text"},
    "po_number": {"type": "text", "instructions": "Purchase order number"},
}
invoice_result = model.extract(
    state=invoice, fields=fields, options={"return_state_text": True}
)
print(json.dumps(invoice_result["answers"], indent=2))

## Inspect values and source evidence

Each answer includes its value and confidence. Single-value fields also report `p_null`, the probability that no value applies. Text extracted from the state has character offsets; list answers carry one span per item. Calculated dates and amounts need not have text spans. Confidence does not guarantee correctness.

The response also preserves truncation and token usage. `reference_date` can anchor dates without a year; native `options` can request quantiles or returned state text. For asynchronous REST calls, use `await model.aextract(...)` with the same arguments.


In [ ]:
# MARK: - SOURCE EVIDENCE
invoice_number = invoice_result["answers"]["invoice_number"]
print(json.dumps(invoice_number, indent=2))
if invoice_number["value"] is not None:
    start, end = invoice_number["span"]
    print("Source text:", invoice_result["state_text"][start:end])
print("Truncated:", invoice_result["truncated"])
print("Usage:", json.dumps(invoice_result["usage"], indent=2))

## Image-only extraction

This embedded PNG is the same labeled image used by the repository's vision tests. Display it, then ask for its ticker, timeframe, and date with an empty text state. Clairic accepts embedded PNG/JPEG/WebP data URLs or `content_type`/`base64` objects.

Image-derived text can have `source="image"`, `span=None`, and `spell_confidence`; the adapter preserves these fields. Inspect the returned image-token usage as well as the answers.


In [ ]:
# MARK: - IMAGE
image_base64 = (
    "iVBORw0KGgoAAAANSUhEUgAAArwAAAEECAIAAABFqXCRAAAUoUlEQVR4nO3WS44kOw5E0bf/TXfP"
    "alBIFIg0NxpJ3TNWiD95gP/9DwAAoOC/dAIAAGAHlgYAAFDC0gAAAEpYGgAAQAlLAwAAKGFpAAAA"
    "JSwNAACghKUBAACUsDQAAIASlgYAAFDC0gAAAEpYGgAAQAlLAwAAKGFpAAAAJSwNAACghKUBAACU"
    "sDQAAIASlgYAAFDC0gAAAEpYGgAAQAlLAwAAKGFpAAAAJSwNAACghKUBAACUsDQAAIASlgYAAFDC"
    "0gAAAEpYGgAAQAlLAwAAKGFpAAAAJSwNAACghKUBAACUsDQAAIASlgYAAFBiXBr+6zIweltK9UC/"
    "GeGv7nSUmY0utu5l2c5/Mv1KSm3Jt5XpqAg3sDRYorelVA/0mxH+6k5HmdnoYutelu38J9OvpNSW"
    "fFuZjopwA0uDJXpbSvVAvxnhr+50lJmNLrbuZdnOfzL9SkptybeV6agIN7A0WKK3pVQP9JsR/upO"
    "R5nZ6GLrXpbt/CfTr6TUlnxbmY6KcANLgyV6W0r1QL8Z4a/udJSZjS627mXZzn8y/UpKbcm3lemo"
    "CDewNFiit6VUD/SbEf7qTkeZ2ehi616W7fwn06+k1JZ8W5mOinADS4MleltK9UC/GeGv7nSUmY0u"
    "tu5l2c5/Mv1KSm3Jt5XpqAg3jHgH2QfqiL66ouzPRVumme2SQ7ZLA1+dmPyW2vGaEW9r4Ac/8M62"
    "6Nmfi7ZMM9slh2yXBr46MfktteM1I97WwA9+4J1t0bM/F22ZZrZLDtkuDXx1YvJbasdrRrytgR/8"
    "wDvbomd/LtoyzWyXHLJdGvjqxOS31I7XjHhbAz/4gXe2Rc/+XLRlmtkuOWS7NPDViclvqR2vGfG2"
    "Bn7wA+9si579uWjLNLNdcsh2aeCrE5PfUjteM+JtDfzgB97ZFj37c9GWaWa75JDt0sBXJya/pXa8"
    "ZsTbGvjBD7zTEf3l2u/dmZXt0pZXlz0J6Ea8rS0ffPZOR/SXa793Z1a2S1teXfYkoBvxtrZ88Nk7"
    "HdFfrv3enVnZLm15ddmTgG7E29rywWfvdER/ufZ7d2Zlu7Tl1WVPAroRb2vLB5+90xH95drv3ZmV"
    "7dKWV5c9CehGvK0tH3z2Tkf0l2u/d2dWtktbXl32JKAb8ba2fPDZOx3RX6793p1Z2S5teXXZk4Bu"
    "xNva8sFn7xSjt6VUj+7IMxv9ZY7O18eRHZwjT/Gk6LPW4JwRjyP7ah3RB1bUllI9uiPPbPSXOTpf"
    "H0d2cI48xZOiz1qDc0Y8juyrdUQfWFFbSvXojjyz0V/m6Hx9HNnBOfIUT4o+aw3OGfE4sq/WEX1g"
    "RW0p1aM78sxGf5mj8/VxZAfnyFM8KfqsNThnxOPIvlpH9IEVtaVUj+7IMxv9ZY7O18eRHZwjT/Gk"
    "6LPW4JwRjyP7ah3RB1bUllI9uiPPbPSXOTpfH0d2cI48xZOiz1qDc0Y8juyrdUQfWFFbSvXojjyz"
    "0V/m6Hx9HNnBOfIUT4o+aw3OGfE4sq/WEX1LRdQ+P88tsm/p3jS31I7XjHhbL3/wDtm/m5drd+S5"
    "RfYt3ZvmltrxmhFv6+UP3iH7d/Ny7Y48t8i+pXvT3FI7XjPibb38wTtk/25ert2R5xbZt3Rvmltq"
    "x2tGvK2XP3iH7N/Ny7U78twi+5buTXNL7XjNiLf18gfvkP27ebl2R55bZN/SvWluqR2vGfG2Xv7g"
    "HbJ/Ny/X7shzi+xbujfNLbXjNSPe1ssfvIMYPftz0Zbks11ycFRUv3P1qxPvvPeWMNmIt/XyB+8g"
    "Rs/+XLQl+WyXHBwV1e9c/erEO++9JUw24m29/ME7iNGzPxdtST7bJQdHRfU7V7868c57bwmTjXhb"
    "L3/wDmL07M9FW5LPdsnBUVH9ztWvTrzz3lvCZCPe1ssfvIMYPftz0Zbks11ycFRUv3P1qxPvvPeW"
    "MNmIt/XyB+8gRs/+XLQl+WyXHBwV1e9c/erEO++9JUw24m29/ME7iNGzPxdtST7bJQdHRfU7V786"
    "8c57bwmTjXhbbZ+cQzalbD/byhSjt9X+soGd/927+reBtTvKdFSEG0a8A8cDHfghtUUX8xTvdJQp"
    "Rm+r/WUDO/+7d/VvA2t3lOmoCDeMeAeOBzrwQ2qLLuYp3ukoU4zeVvvLBnb+d+/q3wbW7ijTURFu"
    "GPEOHA904IfUFl3MU7zTUaYYva32lw3s/O/e1b8NrN1RpqMi3DDiHTge6MAPqS26mKd4p6NMMXpb"
    "7S8b2Pnfvat/G1i7o0xHRbhhxDtwPNCBH1JbdDFP8U5HmWL0ttpfNrDzv3tX/zawdkeZjopww4h3"
    "4HigAz+ktuhinuKdjjLF6G21v2xg53/3rv5tYO2OMh0V4QbeAQAAKGFpAAAAJSwNAACghKUBAACU"
    "sDQAAIASlgYAAFDC0gAAAEpYGgAAQAlLAwAAKGFpAAAAJSwNAACghKUBAACUsDQAAIASlgYAAFDC"
    "0gAAAEpYGgAAQAlLAwAAKGFpAAAAJSwNAACghKUBAACUsDQAAIASlgYAAFDC0gAAAEpYGgAAQAlL"
    "AwAAKGFpAAAAJSwNAACghKUBAACUsDQAAIASlgYAAFDC0gAAAEqMS8N/XbZEd/RzS+2rozsG54je"
    "lryjzDqxokdsaTIjXoeloS+6o59bal8d3TE4R/S25B1l1okVPWJLkxnxOiwNfdEd/dxS++rojsE5"
    "orcl7yizTqzoEVuazIjXYWnoi+7o55baV0d3DM4RvS15R5l1YkWP2NJkRrwOS0NfdEc/t9S+Orpj"
    "cI7obck7yqwTK3rEliYz4nVYGvqiO/q5pfbV0R2Dc0RvS95RZp1Y0SO2NJkRr8PS0Bfd0c8tta+O"
    "7hicI3pb8o4y68SKHrGlyYx4nRHjufdqxY/T8XOxorp70bOdF6Nnfy7W7rAlzzpHRW133hvHeSPG"
    "k303Az8Px8/FiuruRc92Xoye/blYu8OWPOscFbXdeW8c540YT/bdDPw8HD8XK6q7Fz3beTF69udi"
    "7Q5b8qxzVNR2571xnDdiPNl3M/DzcPxcrKjuXvRs58Xo2Z+LtTtsybPOUVHbnffGcd6I8WTfzcDP"
    "w/FzsaK6e9GznRejZ38u1u6wJc86R0Vtd94bx3kjxpN9NwM/D8fPxYrq7kXPdl6Mnv25WLvDljzr"
    "HBW13XlvHOeNGE/23Qz8PBw/Fyuquxc923kxevbnYu0OW/Ksc1TUdue9cZw3YjzZd5P95LInHe5F"
    "z3beET17MmtLnnWOisQ77z0b/DFiPNl3k/3ksicd7kXPdt4RPXsya0uedY6KxDvvPRv8MWI82XeT"
    "/eSyJx3uRc923hE9ezJrS551jorEO+89G/wxYjzZd5P95LInHe5Fz3beET17MmtLnnWOisQ77z0b"
    "/DFiPNl3k/3ksicd7kXPdt4RPXsya0uedY6KxDvvPRv8MWI82XeT/eSyJx3uRc923hE9ezJrS551"
    "jorEO+89G/wxYjzZd5P95LInHe5Fz3beET17MmtLnnWOisQ77z0b/DFiPNl3k/3k2k6KxNqJ3pDS"
    "wJOieudf5midOPds8rAaMZ7su9nyyYknRWLtRG9IaeBJUb3zL3O0Tpx7NnlYjRhP9t1s+eTEkyKx"
    "dqI3pDTwpKje+Zc5WifOPZs8rEaMJ/tutnxy4kmRWDvRG1IaeFJU7/zLHK0T555NHlYjxpN9N1s+"
    "OfGkSKyd6A0pDTwpqnf+ZY7WiXPPJg+rEePJvpstn5x4UiTWTvSGlAaeFNU7/zJH68S5Z5OH1Yjx"
    "ZN/Nlk9OPCkSayd6Q0oDT4rqnX+Zo3Xi3LPJw2rEeLLvJvvJOe68109HdOY+P7rDljzrHDMSTzp8"
    "1i9oRkwi+0Qc0bN33uunIzpznx/dYUuedY4ZiScdPusXNCMmkX0ijujZO+/10xGduc+P7rAlzzrH"
    "jMSTDp/1C5oRk8g+EUf07J33+umIztznR3fYkmedY0biSYfP+gXNiElkn4gjevbOe/10RGfu86M7"
    "bMmzzjEj8aTDZ/2CZsQksk/EET17571+OqIz9/nRHbbkWeeYkXjS4bN+QTNiEtkn4oievfNePx3R"
    "mfv86A5b8qxzzEg86fBZv6AZMYnsE3FEz955r59t0bM/F++8F91hS551YkXZhtwbx3kjxnPv1Wbv"
    "vNfPtujZn4t33ovusCXPOrGibEPujeO8EeO592qzd97rZ1v07M/FO+9Fd9iSZ51YUbYh98Zx3ojx"
    "3Hu12Tvv9bMtevbn4p33ojtsybNOrCjbkHvjOG/EeO692uyd9/rZFj37c/HOe9EdtuRZJ1aUbci9"
    "cZw3Yjz3Xm32znv9bIue/bl4573oDlvyrBMryjbk3jjOGzGee682e+e9frZFz/5cvPNedIctedaJ"
    "FWUbcm8c540Yj+Pd/HinQ7aiR2qvc0TP1r46uoNY0SPEfrZ1nhGvM2I8jndT/2ZE2Yoeqb3OET1b"
    "++roDmJFjxD72dZ5RrzOiPE43k39mxFlK3qk9jpH9Gztq6M7iBU9QuxnW+cZ8TojxuN4N/VvRpSt"
    "6JHa6xzRs7Wvju4gVvQIsZ9tnWfE64wYj+Pd1L8ZUbaiR2qvc0TP1r46uoNY0SPEfrZ1nhGvM2I8"
    "jndT/2ZE2Yoeqb3OET1b++roDmJFjxD72dZ5RrzOiPE43k39mxFlK3qk9jpH9Gztq6M7iBU9Quxn"
    "W+cZ8TqMBwAAlLA0AACAEpYGAABQwtIAAABKWBoAAEAJSwMAAChhaQAAACUsDQAAoISlAQAAlLA0"
    "AACAEpYGAABQwtIAAABKWBoAAEAJSwMAAChhaQAAACUsDQAAoISlAQAAlLA0AACAEpYGAABQwtIA"
    "AABKWBoAAEAJSwMAAChhaQAAACUsDQAAoISlAQAAlLA0AACAEpYGAABQwtIAAABKWBoAAEAJSwMA"
    "AChhaQAAACXGpeE/g2xK2drrKWWj4y/ZJhM9FT2LzqcMrP3bQCwNluifFPvrlLLR8Zdsk4meip5F"
    "51MG1v5tIJYGS/RPiv11Stno+Eu2yURPRc+i8ykDa/82EEuDJfonxf46pWx0/CXbZKKnomfR+ZSB"
    "tX8biKXBEv2TYn+dUjY6/pJtMtFT0bPofMrA2r8NxNJgif5Jsb9OKRsdf8k2meip6Fl0PmVg7d8G"
    "YmmwRP+k2F+nlI2Ov2SbTPRU9Cw6nzKw9m8DdS8NbXdmo9+702FLnnXZaRL92+hb0PmUbOfrP/82"
    "T5YGS/R7dzpsybMuO02ifxt9Czqfku18/eff5snSYIl+706HLXnWZadJ9G+jb0HnU7Kdr//82zxZ"
    "GizR793psCXPuuw0if5t9C3ofEq28/Wff5snS4Ml+r07HbbkWZedJtG/jb4FnU/Jdr7+82/zZGmw"
    "RL93p8OWPOuy0yT6t9G3oPMp2c7Xf/5tniwNluj37nTYkmdddppE/zb6FnQ+Jdv5+s+/zZOlwRL9"
    "3p0OW/LManvzP/48O6O22lfLzv3lzjs4+vntnSwNluj37nTYkmdW25v/8efZGbXVvlp27i933sHR"
    "z2/vZGmwRL93p8OWPLPa3vyPP8/OqK321bJzf7nzDo5+fnsnS4Ml+r07HbbkmdX25n/8eXZGbbWv"
    "lp37y513cPTz2ztZGizR793psCXPrLY3/+PPszNqq3217Nxf7ryDo5/f3snSYIl+706HLXlmtb35"
    "H3+enVFb7atl5/5y5x0c/fz2TpYGS/R7dzpsyTOr7c3/+PPsjNpqXy0795c77+Do57d3vrU0OLRV"
    "JN6ZLRMVYufrP3ecFDlqF2Wj11MaeFK0pfOipYFYGlRtFYl3ZstEhdj5+s8dJ0WO2kXZ6PWUBp4U"
    "bem8aGkglgZVW0XindkyUSF2vv5zx0mRo3ZRNno9pYEnRVs6L1oaiKVB1VaReGe2TFSIna//3HFS"
    "5KhdlI1eT2ngSdGWzouWBmJpULVVJN6ZLRMVYufrP3ecFDlqF2Wj11MaeFK0pfOipYFYGlRtFYl3"
    "ZstEhdj5+s8dJ0WO2kXZ6PWUBp4Ubem8aGkglgZVW0XindkyUSF2vv5zx0mRo3ZRNno9pYEnRVs6"
    "L1oa6K2loS36vTsdtuTp0DZ3MdCW97nlTofs3F/ufF22om+jszRYot+702FLng5tcxcDbXmfW+50"
    "yM795c7XZSv6NjpLgyX6vTsdtuTp0DZ3MdCW97nlTofs3F/ufF22om+jszRYot+702FLng5tcxcD"
    "bXmfW+50yM795c7XZSv6NjpLgyX6vTsdtuTp0DZ3MdCW97nlTofs3F/ufF22om+jszRYot+702FL"
    "ng5tcxcDbXmfW+50yM795c7XZSv6NjpLgyX6vTsdtuTp0DZ3MdCW97nlTofs3F/ufF22om+jszRY"
    "ot+702FLnqK2Mge+pZdrb5Ot/eXO/2hg8t+mxNJgiX7vTocteYrayhz4ll6uvU229pc7/6OByX+b"
    "EkuDJfq9Ox225ClqK3PgW3q59jbZ2l/u/I8GJv9tSiwNluj37nTYkqeorcyBb+nl2ttka3+58z8a"
    "mPy3KbE0WKLfu9NhS56itjIHvqWXa2+Trf3lzv9oYPLfpsTSYIl+706HLXmK2soc+JZerr1NtvaX"
    "O/+jgcl/mxJLgyX6vTsdtuQpaitz4Ft6ufY22dpf7vyPBib/bUrdS4NIjO6oaMudDmKe9wzs/JZA"
    "A1OqBxooW/sjnRfLdDSkIRBLg1rRljsdxDzvGdj5LYEGplQPNFC29kc6L5bpaEhDIJYGtaItdzqI"
    "ed4zsPNbAg1MqR5ooGztj3ReLNPRkIZALA1qRVvudBDzvGdg57cEGphSPdBA2dof6bxYpqMhDYFY"
    "GtSKttzpIOZ5z8DObwk0MKV6oIGytT/SebFMR0MaArE0qBVtudNBzPOegZ3fEmhgSvVAA2Vrf6Tz"
    "YpmOhjQEYmlQK9pyp4OY5z0DO78l0MCU6oEGytb+SOfFMh0NaQi0ZjwAACCLpQEAAJSwNAAAgBKW"
    "BgAAUMLSAAAASlgaAABACUsDAAAoYWkAAAAlLA0AAKCEpQEAAJSwNAAAgBKWBgAAUMLSAAAASlga"
    "AABACUsDAAAoYWkAAAAl/wcRkr3M3ZNA1AAAAABJRU5ErkJggg=="
)
display(Image(data=base64.b64decode(image_base64)))

In [ ]:
# MARK: - IMAGE EXTRACTION
image_fields = {
    "ticker": {"type": "text", "instructions": "Ticker printed under TICKER"},
    "timeframe": {
        "type": "text",
        "instructions": "Timeframe printed under TIMEFRAME",
    },
    "date": {"type": "date", "instructions": "Date printed under DATE"},
}
image_result = model.extract(
    state="",
    images=[{"content_type": "image/png", "base64": image_base64}],
    fields=image_fields,
)
print(json.dumps(image_result["answers"], indent=2))
print("Usage:", json.dumps(image_result["usage"], indent=2))

## Combine Clairic and Clef into nested Pydantic output

The shared [Worker example](../libs/langchain-cloudflare/examples/workers/src/system_one.py) calls the same production model classes: Clairic extracts candidate companies, Clef decides eligibility and roles and selects a source line, then Clairic extracts printed tickers. The function validates the assembled result with the existing `Data`, `Announcement`, and `Entity` Pydantic models.

The two public examples below require three native calls for the partnership and two for the rejected earnings release. Inspect `extracted` for the nested result and `stages` for the native evidence.


In [ ]:
# MARK: - COMBINED EXTRACTION
from examples.workers.src.system_one import extract_announcements

press_releases = {
    "partnership": (
        "Acme Corp (NYSE: ACME) today announced a strategic partnership with "
        "TechGiant Inc to jointly develop next-generation AI solutions.\n"
        "The partnership will combine Acme's expertise in cloud infrastructure "
        "with TechGiant's machine learning capabilities."
    ),
    "earnings": "Apple Inc announced record Q4 earnings, beating analyst expectations.",
}
decision_model = CloudflareWorkersAIDecisionModel(ai_gateway=None)
combined_results = {}
for label, text in press_releases.items():
    combined_results[label] = await extract_announcements(
        text=text, extraction_model=model, decision_model=decision_model
    )
    print(label, json.dumps(combined_results[label]["extracted"], indent=2))

In [ ]:
# MARK: - NATIVE STAGES
for stage, native_result in combined_results["partnership"]["stages"].items():
    print(stage, json.dumps(native_result, indent=2))

The combined example handles one primary announcement, with each context candidate on its own input line and at most 61 candidate company mentions. It cannot recover names omitted by Clairic or convert arbitrary Pydantic schemas.

The retained standalone company-list and full-sentence fixtures still have expected model-quality failures. The combined example passes the announcement fixtures by assigning classification and filtering to Clef.


## Python Worker binding

Inside a Python Worker, the existing AI binding supplies authentication. Use `aextract()` there; Jupyter does not provide `env.AI`.

```python
model = CloudflareWorkersAIExtractionModel(binding=self.env.AI)
result = await model.aextract(state=invoice, fields=fields)
```

The Worker example exposes `/extraction` and `/structured-system-one`. See the [library README](../libs/langchain-cloudflare/README.md#extraction-models-clairic) for the native API options.
